In [2]:
pip install pandas numpy scikit-learn xgboost tensorflow

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/101.7 MB 3.4 MB/s eta 0:00:31
   ---------------------------------------- 1.0/101.7 MB 2.6 MB/s eta 0:00:40
    --------------------------------------- 1.6/101.7 MB 2.4 MB/s eta 0:00:41
    --------------------------------------- 2.4/101.7 MB 2.6 MB/s eta 0:00:38
   - -------------------------------------- 3.1/101.7 MB 2.8 MB/s eta 0:00:35
   - -------------------------------------- 3.7/101.7 MB 2.9 MB/s eta 0:00:35
   - -------------------------------------- 4.5/101.7 MB 2.9 MB/s eta 0:00:34
   - -------------------------------------- 5.0/101.7 MB 2.9 MB/s eta 0:00:34
   -- ------------------------------------- 5.8/101.7 MB 3.0 MB/s eta 0:00:32
   -- ------------------------------------- 6.6/101.7 MB 3.1 MB/s eta 0:00:32
   -- ------------------------------------- 7.3/101.7 MB 3.1 MB/s eta 0:00:31


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.svm import SVC

from xgboost import XGBClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
 


In [3]:
df = pd.read_csv(r"healthcare-dataset-stroke-data.csv")

df.drop("id", axis=1, inplace=True)

# Handle missing values
imputer = SimpleImputer(strategy='mean')
df['bmi'] = imputer.fit_transform(df[['bmi']])

In [4]:
# Encode categorical
le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

In [5]:
X = df.drop("stroke", axis=1)
y = df["stroke"]

In [6]:
scaler = StandardScaler()
X = scaler.fit_transform(X)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [8]:
models = {
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(probability=True),
    "AdaBoost": AdaBoostClassifier(),
    "SGD": SGDClassifier(loss='log_loss'),
    "XGBoost": XGBClassifier(eval_metric='logloss')
}


In [9]:
all_scores = {}

print("Model Accuracy:\n")
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    all_scores[name] = acc  
    print(f"{name}: {acc:.4f}")

Model Accuracy:

Decision Tree: 0.9129
Random Forest: 0.9384
KNN: 0.9393
SVM: 0.9393
AdaBoost: 0.9393
SGD: 0.9393
XGBoost: 0.9374


In [15]:
ann = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [16]:
ann.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
ann.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)

In [17]:
_, ann_acc = ann.evaluate(X_test, y_test, verbose=0)
ann_preds = (ann.predict(X_test) > 0.5).astype(int).flatten()
all_scores['ANN'] = ann_acc
print(f"\nANN Accuracy: {ann_acc:.4f}")


32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step

ANN Accuracy: 0.9393


In [18]:
voting = VotingClassifier(
    estimators=[
        ('dt', DecisionTreeClassifier()),
        ('rf', RandomForestClassifier()),
        ('knn', KNeighborsClassifier()),
        ('svm', SVC(probability=True)),
        ('ada', AdaBoostClassifier()),
        ('sgd', SGDClassifier(loss='log_loss')),
        ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='logloss'))
    ],
    voting='soft'
)

In [19]:
voting.fit(X_train, y_train)
voting_acc = accuracy_score(y_test, voting.predict(X_test))
all_scores['Voting Classifier'] = voting_acc  
print(f"\nVoting Classifier Accuracy: {voting_acc:.4f}")

C:\Users\Azhagulakshmi\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:51:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Voting Classifier Accuracy: 0.9403


In [20]:
proba_list = [m.predict_proba(X_test)[:, 1] for m in models.values()]

In [21]:
ann_proba = ann.predict(X_test).flatten()
proba_list.append(ann_proba)

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [22]:
avg_proba = np.mean(proba_list, axis=0)
combined_preds = (avg_proba > 0.5).astype(int)
combined_acc = accuracy_score(y_test, combined_preds)
all_scores['Voting (8 models incl. ANN)'] = combined_acc
print(f"Voting Classifier (8 models incl. ANN) Accuracy: {combined_acc:.4f}")

Voting Classifier (8 models incl. ANN) Accuracy: 0.9393


In [24]:
print("\n" + "="*50)
print("      MODEL COMPARISON (Best → Worst)")
print("="*50)
for name, score in sorted(all_scores.items(), key=lambda x: -x[1]):
    bar = " " * int(score * 40)
    print(f"{name:<35} {score:.4f}  {bar}")


      MODEL COMPARISON (Best → Worst)
Voting Classifier                   0.9403                                       
KNN                                 0.9393                                       
SVM                                 0.9393                                       
AdaBoost                            0.9393                                       
SGD                                 0.9393                                       
Voting (8 models incl. ANN)         0.9393                                       
ANN                                 0.9393                                       
Random Forest                       0.9384                                       
XGBoost                             0.9374                                       
Decision Tree                       0.9129                                      


In [26]:
best_model = max(all_scores, key=all_scores.get)
print("="*50)
print(f"\nBest Model: {best_model}")
print(f"   Accuracy  : {all_scores[best_model]:.4f}")
print(f"\n Conclusion: '{best_model}' outperforms all other models")
print(   "   for stroke risk prediction on this dataset.")


Best Model: Voting Classifier
   Accuracy  : 0.9403

 Conclusion: 'Voting Classifier' outperforms all other models
   for stroke risk prediction on this dataset.
